# Gold Layer - Dimensional Data Model

This notebook builds the analytics-ready Gold layer of the
Olist E-Commerce Lakehouse.

## Objectives

- Build a star-schema dimensional model
- Define a clear fact-table grain
- Create reusable business dimensions
- Preserve accurate sales measures
- Prevent payment-to-item join inflation
- Create analytics-ready Delta tables
- Validate dimensional relationships and financial measures

## Fact Table Grain

`fact_sales` contains one row per order item.

The natural business key is:

`order_id + order_item_id`

## Gold Tables

### Dimensions
- `dim_customer`
- `dim_product`
- `dim_seller`
- `dim_date`

### Facts
- `fact_sales`
- `fact_payments`

### Analytical Marts
- `order_summary`
- `customer_metrics`
- `product_performance`

In [0]:
from pyspark.sql import functions as F

CATALOG = "ecommerce_lakehouse"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD}"
)

print(f"Source : {SILVER}")
print(f"Target : {GOLD}")

Source : ecommerce_lakehouse.silver
Target : ecommerce_lakehouse.gold


In [0]:
def write_gold(df, table_name):

    target = f"{GOLD}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    count = spark.table(target).count()

    print(
        f"{target:<55} {count:>12,} rows"
    )

    return count

In [0]:
customers = spark.table(
    f"{SILVER}.customers"
)

dim_customer = (
    customers

    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )

    .dropDuplicates(["customer_id"])

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

write_gold(
    dim_customer,
    "dim_customer"
)

ecommerce_lakehouse.gold.dim_customer                         99,441 rows


99441

In [0]:
products = spark.table(
    f"{SILVER}.products"
)

categories = spark.table(
    f"{SILVER}.product_category_translation"
)

dim_product = (
    products.alias("p")

    .join(
        categories.alias("c"),
        on="product_category_name",
        how="left"
    )

    .select(
        F.col("p.product_id"),

        F.col(
            "p.product_category_name"
        ),

        F.coalesce(
            F.col(
                "c.product_category_name_english"
            ),
            F.col(
                "p.product_category_name"
            )
        ).alias(
            "product_category_english"
        ),

        F.col("p.product_name_length"),
        F.col("p.product_description_length"),
        F.col("p.product_photos_qty"),
        F.col("p.product_weight_g"),
        F.col("p.product_length_cm"),
        F.col("p.product_height_cm"),
        F.col("p.product_width_cm")
    )

    .dropDuplicates(["product_id"])

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

write_gold(
    dim_product,
    "dim_product"
)

ecommerce_lakehouse.gold.dim_product                          32,951 rows


32951

In [0]:
sellers = spark.table(
    f"{SILVER}.sellers"
)

dim_seller = (
    sellers

    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    )

    .dropDuplicates(["seller_id"])

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

write_gold(
    dim_seller,
    "dim_seller"
)

ecommerce_lakehouse.gold.dim_seller                            3,095 rows


3095

In [0]:
orders = spark.table(
    f"{SILVER}.orders"
)

date_range = (
    orders
    .select(
        F.min(
            F.to_date("order_purchase_timestamp")
        ).alias("min_date"),

        F.max(
            F.to_date("order_purchase_timestamp")
        ).alias("max_date")
    )
    .first()
)

min_date = date_range["min_date"]
max_date = date_range["max_date"]

print("Minimum order date:", min_date)
print("Maximum order date:", max_date)

Minimum order date: 2016-09-04
Maximum order date: 2018-10-17


In [0]:
dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{min_date}'),
                to_date('{max_date}'),
                interval 1 day
            )
        ) AS full_date
        """
    )

    .withColumn(
        "date_key",
        F.date_format(
            "full_date",
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "year",
        F.year("full_date")
    )

    .withColumn(
        "quarter",
        F.quarter("full_date")
    )

    .withColumn(
        "month",
        F.month("full_date")
    )

    .withColumn(
        "month_name",
        F.date_format(
            "full_date",
            "MMMM"
        )
    )

    .withColumn(
        "week_of_year",
        F.weekofyear("full_date")
    )

    .withColumn(
        "day_of_month",
        F.dayofmonth("full_date")
    )

    .withColumn(
        "day_of_week",
        F.dayofweek("full_date")
    )

    .withColumn(
        "day_name",
        F.date_format(
            "full_date",
            "EEEE"
        )
    )

    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin(
            1, 7
        )
    )

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

write_gold(
    dim_date,
    "dim_date"
)

ecommerce_lakehouse.gold.dim_date                                774 rows


774

In [0]:
order_items = spark.table(
    f"{SILVER}.order_items"
)

orders = spark.table(
    f"{SILVER}.orders"
)

In [0]:
fact_sales = (
    order_items.alias("i")

    .join(
        orders.alias("o"),
        on="order_id",
        how="inner"
    )

    .select(
        F.col("i.order_id"),
        F.col("i.order_item_id"),

        F.col("o.customer_id"),
        F.col("i.product_id"),
        F.col("i.seller_id"),

        F.date_format(
            F.to_date(
                F.col(
                    "o.order_purchase_timestamp"
                )
            ),
            "yyyyMMdd"
        ).cast("int").alias(
            "purchase_date_key"
        ),

        F.col("o.order_status"),

        F.col(
            "o.order_purchase_timestamp"
        ),

        F.col(
            "o.order_approved_at"
        ),

        F.col(
            "o.order_delivered_customer_date"
        ),

        F.col(
            "o.order_estimated_delivery_date"
        ),

        F.col("i.price"),
        F.col("i.freight_value"),
        F.col("i.item_total_value")
    )

    .withColumn(
        "delivery_days",
        F.datediff(
            F.to_date(
                "order_delivered_customer_date"
            ),
            F.to_date(
                "order_purchase_timestamp"
            )
        )
    )

    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.to_date(
                "order_delivered_customer_date"
            ),
            F.to_date(
                "order_estimated_delivery_date"
            )
        )
    )

    .withColumn(
        "is_late_delivery",
        F.when(
            F.col(
                "order_delivered_customer_date"
            ).isNull(),
            F.lit(None).cast("boolean")
        )
        .otherwise(
            F.col("delivery_delay_days") > 0
        )
    )

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

In [0]:
print(
    "Silver order-item rows:",
    order_items.count()
)

print(
    "Gold fact rows:",
    fact_sales.count()
)

Silver order-item rows: 112650
Gold fact rows: 112650


In [0]:
write_gold(
    fact_sales,
    "fact_sales"
)

ecommerce_lakehouse.gold.fact_sales                          112,650 rows


112650

In [0]:
payments = spark.table(
    f"{SILVER}.order_payments"
)

fact_payments = (
    payments.alias("p")

    .join(
        orders
        .select(
            "order_id",
            "customer_id",
            "order_purchase_timestamp"
        )
        .alias("o"),

        on="order_id",
        how="inner"
    )

    .select(
        F.col("p.order_id"),
        F.col("p.payment_sequential"),

        F.col("o.customer_id"),

        F.date_format(
            F.to_date(
                "o.order_purchase_timestamp"
            ),
            "yyyyMMdd"
        ).cast("int").alias(
            "purchase_date_key"
        ),

        F.col("p.payment_type"),
        F.col("p.payment_installments"),
        F.col("p.payment_value")
    )

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

write_gold(
    fact_payments,
    "fact_payments"
)

ecommerce_lakehouse.gold.fact_payments                       103,886 rows


103886

In [0]:
order_item_summary = (
    fact_sales

    .groupBy("order_id")

    .agg(
        F.count("*")
        .alias("item_count"),

        F.sum("price")
        .alias("product_value"),

        F.sum("freight_value")
        .alias("freight_value"),

        F.sum("item_total_value")
        .alias("order_item_value")
    )
)

In [0]:
payment_summary = (
    fact_payments

    .groupBy("order_id")

    .agg(
        F.sum("payment_value")
        .alias("payment_value"),

        F.count("*")
        .alias("payment_count"),

        F.max("payment_installments")
        .alias("max_installments")
    )
)

In [0]:
order_summary = (
    orders.alias("o")

    .join(
        order_item_summary.alias("i"),
        on="order_id",
        how="left"
    )

    .join(
        payment_summary.alias("p"),
        on="order_id",
        how="left"
    )

    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",

        "item_count",
        "product_value",
        "freight_value",
        "order_item_value",

        "payment_value",
        "payment_count",
        "max_installments"
    )

    .withColumn(
        "delivery_days",
        F.datediff(
            F.to_date(
                "order_delivered_customer_date"
            ),
            F.to_date(
                "order_purchase_timestamp"
            )
        )
    )

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

write_gold(
    order_summary,
    "order_summary"
)

ecommerce_lakehouse.gold.order_summary                        99,441 rows


99441

In [0]:
customer_metrics = (
    order_summary.alias("o")

    .join(
        dim_customer
        .select(
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        )
        .alias("c"),

        on="customer_id",
        how="inner"
    )

    .groupBy(
        "customer_unique_id"
    )

    .agg(
        F.countDistinct("order_id")
        .alias("total_orders"),

        F.sum("payment_value")
        .alias("total_spend"),

        F.avg("payment_value")
        .alias("average_order_value"),

        F.min("order_purchase_timestamp")
        .alias("first_order_at"),

        F.max("order_purchase_timestamp")
        .alias("last_order_at"),

        F.first(
            "customer_city",
            ignorenulls=True
        ).alias("customer_city"),

        F.first(
            "customer_state",
            ignorenulls=True
        ).alias("customer_state")
    )

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

write_gold(
    customer_metrics,
    "customer_metrics"
)

ecommerce_lakehouse.gold.customer_metrics                     96,096 rows


96096

In [0]:
product_performance = (
    fact_sales.alias("f")

    .join(
        dim_product
        .select(
            "product_id",
            "product_category_english"
        )
        .alias("p"),

        on="product_id",
        how="left"
    )

    .groupBy(
        "product_id",
        "product_category_english"
    )

    .agg(
        F.count("*")
        .alias("units_sold"),

        F.countDistinct("order_id")
        .alias("order_count"),

        F.sum("price")
        .alias("product_revenue"),

        F.sum("freight_value")
        .alias("freight_revenue"),

        F.sum("item_total_value")
        .alias("gross_item_value"),

        F.avg("price")
        .alias("average_item_price")
    )

    .withColumn(
        "_gold_processed_at",
        F.current_timestamp()
    )
)

write_gold(
    product_performance,
    "product_performance"
)

ecommerce_lakehouse.gold.product_performance                  32,951 rows


32951

In [0]:
duplicate_fact_keys = (
    spark.table(
        f"{GOLD}.fact_sales"
    )

    .groupBy(
        "order_id",
        "order_item_id"
    )

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)

print(
    "Duplicate fact_sales keys:",
    duplicate_fact_keys
)

Duplicate fact_sales keys: 0


In [0]:
fact = spark.table(
    f"{GOLD}.fact_sales"
)

relationship_tests = {
    "missing_customer":
        fact.join(
            dim_customer.select(
                "customer_id"
            ),
            on="customer_id",
            how="left_anti"
        ).count(),

    "missing_product":
        fact.join(
            dim_product.select(
                "product_id"
            ),
            on="product_id",
            how="left_anti"
        ).count(),

    "missing_seller":
        fact.join(
            dim_seller.select(
                "seller_id"
            ),
            on="seller_id",
            how="left_anti"
        ).count()
}

display(
    spark.createDataFrame([
        {
            "test": name,
            "failed_rows": value,
            "status":
                "PASS"
                if value == 0
                else "FAIL"
        }
        for name, value
        in relationship_tests.items()
    ])
)

failed_rows,status,test
0,PASS,missing_customer
0,PASS,missing_product
0,PASS,missing_seller


In [0]:
silver_item_total = (
    spark.table(
        f"{SILVER}.order_items"
    )
    .agg(
        F.sum("item_total_value")
    )
    .first()[0]
)

gold_item_total = (
    spark.table(
        f"{GOLD}.fact_sales"
    )
    .agg(
        F.sum("item_total_value")
    )
    .first()[0]
)

silver_payment_total = (
    spark.table(
        f"{SILVER}.order_payments"
    )
    .agg(
        F.sum("payment_value")
    )
    .first()[0]
)

gold_payment_total = (
    spark.table(
        f"{GOLD}.fact_payments"
    )
    .agg(
        F.sum("payment_value")
    )
    .first()[0]
)

validation = [
    {
        "measure": "item_total_value",
        "silver_total": float(
            silver_item_total
        ),
        "gold_total": float(
            gold_item_total
        ),
        "difference": float(
            gold_item_total
            - silver_item_total
        )
    },
    {
        "measure": "payment_value",
        "silver_total": float(
            silver_payment_total
        ),
        "gold_total": float(
            gold_payment_total
        ),
        "difference": float(
            gold_payment_total
            - silver_payment_total
        )
    }
]

display(
    spark.createDataFrame(validation)
)

difference,gold_total,measure,silver_total
0.0,1.584355324E7,item_total_value,1.584355324E7
0.0,1.600887212E7,payment_value,1.600887212E7


In [0]:
GOLD_TABLES = [
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_date",
    "fact_sales",
    "fact_payments",
    "order_summary",
    "customer_metrics",
    "product_performance"
]

summary = []

for table_name in GOLD_TABLES:

    df = spark.table(
        f"{GOLD}.{table_name}"
    )

    summary.append({
        "table": table_name,
        "rows": df.count(),
        "columns": len(df.columns)
    })

display(
    spark.createDataFrame(summary)
    .orderBy("table")
)

columns,rows,table
9,96096,customer_metrics
6,99441,dim_customer
12,774,dim_date
11,32951,dim_product
5,3095,dim_seller
8,103886,fact_payments
18,112650,fact_sales
15,99441,order_summary
9,32951,product_performance
